In [1]:
from pathlib import Path

import rioxarray as rio

# Convert units

Espen would like all grids converting from mg/m2 to meq/m2.

In [2]:
base_path = r"/home/jovyan/shared/common/critical_loads/raster/for_espen"
ndv = -9999

for path in Path(base_path).rglob("*.tif"):
    if "100smgn" in path.name:
        # Sat veg. N only
        print(path.name)
        new_name = path.name.replace("100smgn", "meqn")
        out_path = path.parent.joinpath(new_name)

        # Convert
        da = rio.open_rasterio(path, mask_and_scale=True).sel(band=1)
        da = 100 * da / 14
        da = da.fillna(ndv)#.round(0)
        da.rio.write_nodata(ndv, inplace=True)
        da.rio.to_raster(out_path, compress="lzw", dtype="float32")

    elif "mg" in path.name:
        print(path.name)
        new_name = path.name.replace("mg", "meq")
        out_path = path.parent.joinpath(new_name)

        # Determine parameter to convert
        if "veg" in path.name:
            par = "n"
        elif "ndep" in path.name:
            par = "n"
        elif "sdep" in path.name:
            par = "s"
        else:
            raise ValueError("Could not identify parameter.")

        # Convert
        da = rio.open_rasterio(path, mask_and_scale=True).sel(band=1)
        if par == "s":
            da = 2 * da / 32.06
        else:
            da = da / 14
        da = da.round(0)
        da = da.fillna(ndv)#.round(0)
        da.rio.write_nodata(ndv, inplace=True)
        da.rio.to_raster(out_path, compress="lzw", dtype="float32")

sat_veg_30m_cl_100smgn_m2_yr_subdiv_water.tif
